# Exploratory Data Analysis — 30-Day Readmission

Dataset: **UCI Diabetes 130-US Hospitals (1999-2008)** (~101K encounters).
This notebook documents the data understanding that motivated the feature
engineering in `src/readmission/features.py`.

Run `make data` first to populate the local cache.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from readmission.config import load_config
from readmission.data import load_raw, make_target
from readmission.features import engineer_features

cfg = load_config()
raw = load_raw(cfg)
raw = raw.replace('?', np.nan)
print(raw.shape)
raw.head()

## Target balance

The clinically meaningful target is readmission **within 30 days** (`<30`).
It is the minority class — which is why PR-AUC and calibration matter more
than raw accuracy.

In [ ]:
print(raw['readmitted'].value_counts())
y = make_target(raw, positive=cfg.target_positive_label)
print(f"Positive (<30) rate: {y.mean():.3f}")
y.value_counts().plot(kind='bar', title='Readmitted within 30 days (1=yes)')
plt.show()

## Missingness

Several columns use `?` as a missing marker. `weight`, `payer_code`, and
`medical_specialty` are very sparse and are dropped in config.

In [ ]:
missing = raw.isna().mean().sort_values(ascending=False)
missing[missing > 0].head(15)

## Diagnosis grouping

Raw ICD-9 codes are high-cardinality. `engineer_features` collapses
`diag_1/2/3` into clinical categories.

In [ ]:
feats = engineer_features(raw, drop_columns=cfg.drop_columns)
print('Engineered columns:', feats.shape[1])
feats['diag_1_group'].value_counts().plot(kind='barh', title='Primary diagnosis group')
plt.tight_layout(); plt.show()

## Derived numeric features vs. target

Prior-utilization signals (`total_visits`, `number_inpatient`) and medication
complexity are expected to carry signal for readmission.

In [ ]:
numeric = feats.select_dtypes('number').copy()
numeric['target'] = y.values
corr = numeric.corr()['target'].drop('target').sort_values(key=np.abs, ascending=False)
corr.head(15)